# StatTest-Pro: End-to-End A/B Testing Example
This notebook demonstrates how to calculate sample sizes, run safety checks, evaluate an experiment, and output an executive report.

In [1]:
%load_ext autoreload
%autoreload 2

from stattest_pro.power import calculate_sample_size
from stattest_pro.srm import check_srm, SRMError
from stattest_pro.evaluator import evaluate_conversion
from stattest_pro.reporter import generate_report
import random

### 1. Power Calculation
Compute required sample size based on a 5% baseline rate and a desired Minimum Detectable Effect (MDE) of +1% absolute.

In [2]:
mde = 0.01
alpha = 0.05
power = 0.80
required_n = calculate_sample_size(baseline_rate=0.05, mde=mde, alpha=alpha, power=power)
print(f"Target Sample Size Per Variant: {required_n}")

Target Sample Size Per Variant: 8143


### 2. Mocking Data & Checking Invariants (SRM)

In [3]:
# Simulating realistic experiment traffic (slightly off 50/50 but within random variance)
total_a = required_n + random.randint(-50, 50)
total_b = required_n + random.randint(-50, 50)

conversions_a = int(total_a * 0.050)
conversions_b = int(total_b * 0.062) # We achieved a +1.2% absolute lift

try:
    p_srm = check_srm(total_a, total_b)
    print(f"SRM Check Passed (p={p_srm:.4f})")
except SRMError as e:
    print(f"Experiment Halted: {e}")

SRM Check Passed (p=0.9312)


### 3. Evaluating Results & Generating Report

In [4]:
results = evaluate_conversion(
    conversions_a, total_a,
    conversions_b, total_b,
    alpha=alpha, mde=mde
)

print(f"Decision: {results['decision']}")
print(f"P-Value: {results['p_value']:.5f}")

report_path = generate_report(results, output_file="../reports/example_report.html")
print(f"Executive report created at: {report_path}")

Decision: Launch (Statistically significant and exceeds MDE)
P-Value: 0.00088
Executive report created at: D:\00_projects\StatTest-Pro\reports\example_report.html
